In [ ]:
# Odkomentuj i uruchom tę linijkę, jeśli nie masz zainstalowanych pakietów rel
# !pip install pytorch-lightning wandb optuna torchvision

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import pytorch_lightning as pl
import wandb

In [ ]:
class STL10VRAMDataModule(pl.LightningDataModule):
    def __init__(self, data_dir='./data', batch_size=256):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        # Upewniamy się, że używamy GPU
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def prepare_data(self):
        # Ta funkcja wywoła się tylko raz, żeby pobrać dane na dysk
        datasets.STL10(self.data_dir, split='train', download=True)
        datasets.STL10(self.data_dir, split='test', download=True)

    def setup(self, stage=None):
        # Transformacja: konwersja do tensora i normalizacja (dla obrazków 96x96)
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

        # Ładujemy zbiór treningowy...
        train_set = datasets.STL10(self.data_dir, split='train', transform=transform)
        # ...i MAGICZNY TRIK: pakujemy wszystko od razu na GPU!
        # Chwilę to potrwa przy pierwszym uruchomieniu, ale potem trening poleci błyskawicznie.
        x_train = torch.stack([x for x, y in train_set]).to(self.device)
        y_train = torch.tensor([y for x, y in train_set]).to(self.device)
        self.train_dataset = TensorDataset(x_train, y_train)

        # To samo dla zbioru testowego (użyjemy go do walidacji)
        val_set = datasets.STL10(self.data_dir, split='test', transform=transform)
        x_val = torch.stack([x for x, y in val_set]).to(self.device)
        y_val = torch.tensor([y for x, y in val_set]).to(self.device)
        self.val_dataset = TensorDataset(x_val, y_val)

    def train_dataloader(self):
        # num_workers=0 i pin_memory=False, bo dane już siedzą w karcie graficznej!
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

In [ ]:
class SimpleCNN(pl.LightningModule):
    def __init__(self, learning_rate=1e-3, hidden_size=128):
        super().__init__()
        # Zapisujemy hiperparametry (WandB automatycznie je zaloguje)
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        
        # Architektura modelu (obrazki wejściowe to 3 kanały RGB, 96x96 pikseli)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        
        # Po dwóch poolingach 96x96 zmniejszy się do 24x24. 32 to liczba kanałów.
        self.fc1 = nn.Linear(32 * 24 * 24, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 10) # 10 klas w STL-10
        
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Spłaszczanie
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        # Logujemy loss na WandB
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        # Liczymy dokładność (accuracy)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Logujemy metryki walidacyjne
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [ ]:
import wandb
# To poprosi Cię o wklejenie klucza API z Twojego konta wandb.ai
wandb.login()

In [ ]:
import optuna
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
import warnings

# Wyłączamy zbędne ostrzeżenia
warnings.filterwarnings("ignore")

def objective(trial):
    # 1. Optuna losuje nam parametry do przetestowania
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])

    # 2. Inicjalizacja modelu i naszego DataModule (z trikiem na VRAM)
    model = SimpleCNN(learning_rate=lr, hidden_size=hidden_size)
    datamodule = STL10VRAMDataModule(batch_size=256)

    # 3. Inicjalizacja WandB Loggera!
    # Każda próba Optuny dostanie własną nazwę, żeby łatwo je było porównać
    wandb_logger = WandbLogger(
        project="HW1-Optuna-STL10", # Nazwa projektu na Twoim profilu WandB
        name=f"trial_{trial.number}"
    )

    # 4. Konfiguracja Trenera
    trainer = pl.Trainer(
        max_epochs=5, # Zostawiamy 5 epok na próbę do testów
        logger=wandb_logger,
        accelerator='gpu',
        devices=1,
        enable_progress_bar=False,
        enable_model_summary=False
    )

    # 5. Odpalamy trening!
    trainer.fit(model, datamodule=datamodule)

    # 6. BARDZO WAŻNE: Zamykamy sesję WandB, żeby kolejna próba miała czystą kartę
    wandb.finish()

    # 7. Pobieramy wynik walidacji i zwracamy do Optuny
    val_acc = trainer.callback_metrics.get("val_acc")
    return val_acc.item() if val_acc is not None else 0.0

# Tworzymy badanie ("study") w Optunie. Cel: ZMAKSYMALIZOWAĆ accuracy
study = optuna.create_study(direction="maximize")

print("Rozpoczynam szukanie hiperparametrów z WandB...")
study.optimize(objective, n_trials=5)

print("\n--- KONIEC OPTYMALIZACJI ---")
print(f"Najlepsze znalezione parametry: {study.best_params}")
print(f"Najlepsza dokładność: {study.best_value:.4f}")